<a href="https://colab.research.google.com/github/admmrphy/Seed-Dominated-Output-in-Encoder-Decoder-Molecular-Generation/blob/main/run_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MolGen generation — Colab runner

Accompanies `generation.py`. This notebook:
1. installs the pinned dependency stack,
2. mounts Google Drive,
3. checks paths and GPU,
4. runs a small **smoke test**,
5. runs the **full generation** (resumable, one CSV per cell).

**Order of operations:** run the install cell, then **restart the runtime** (Runtime → Restart session) before running anything else. Restarting is required because `torch` is reinstalled.

## 1. Install pinned dependencies

In [ ]:
# Pinned to the project pipeline: transformers 4.40.2 / peft 0.10.0 / torch 2.1.0.
# numpy pinned < 2 because torch 2.1.0 is not compatible with numpy 2.x.
!pip install -q \
    numpy==1.26.4 \
    torch==2.1.0 \
    transformers==4.40.2 \
    peft==0.10.0 \
    selfies==2.1.1 \
    rdkit==2023.9.6 \
    pandas

# Fallbacks if a pin fails to resolve on the current Colab image:
#   - rdkit: replace "rdkit==2023.9.6" with a bare  rdkit
#   - torch CUDA build: add  --index-url https://download.pytorch.org/whl/cu121

print('\nInstall complete. RESTART THE RUNTIME now (Runtime > Restart session),')
print('then continue from section 2. Do NOT run further cells before restarting.')

ERROR: Ignored the following versions that require a different python version: 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Python >=3.7,<3.11
ERROR: Could not find a version that satisfies the requirement torch==2.1.0 (from versions: 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1, 2.10.0, 2.11.0, 2.12.0, 2.12.1, 2.13.0)
ERROR: No matching distribution found for torch==2.1.0

Install complete. RESTART THE RUNTIME now (Runtime > Restart session),
then continue from section 2. Do NOT run further cells before restarting.


## 2. Verify environment (run after restarting)

Confirm the pinned versions took and that a GPU is attached.

In [ ]:
import numpy, torch, transformers, peft, rdkit, selfies, pandas
print('numpy       ', numpy.__version__)
print('torch       ', torch.__version__)
print('transformers', transformers.__version__)
print('peft        ', peft.__version__)
print('rdkit       ', rdkit.__version__)
print('selfies     ', selfies.__version__)
print('pandas      ', pandas.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# Expected: torch 2.1.0, transformers 4.40.2, peft 0.10.0, numpy 1.26.4, CUDA True.

ModuleNotFoundError: No module named 'rdkit'

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Check project paths and list checkpoints

`My Drive` maps to `/content/drive/MyDrive` (no space) once mounted. This lists the checkpoint folders so you can confirm they are named `standard_run0..2`, `weighted_run0..2`, `mpt_standard_run0..2`, `mpt_weighted_run0..2`.

In [ ]:
import os
PROJECT = '/content/drive/MyDrive/MOSESproject'
CKPT = f'{PROJECT}/checkpoints'

checks = [
    PROJECT,
    CKPT,
    f'{PROJECT}/moses_full_properties_with_selfies.csv',
    f'{PROJECT}/generation.py',
]
for p in checks:
    print('OK  ' if os.path.exists(p) else 'MISS', p)

print()
if os.path.isdir(CKPT):
    print('checkpoint folders found:')
    for f in sorted(os.listdir(CKPT)):
        print('   ', f)
else:
    print('checkpoints folder not found at', CKPT)

## 5. Stage `generation.py`

Upload `generation.py` into your `MOSESproject` folder on Drive (or adjust the path below). It is copied to the local runtime and put on the import path so it can be reloaded cleanly.

Also set the held-out split path inside `generation.py` (`MOSES_HELDOUT_CSV`) to your MOSES test split if you want the 'Uniqueness vs Held-out' column populated; otherwise it is written as NaN.

In [ ]:
import sys, shutil, os
PROJECT = '/content/drive/MyDrive/MOSESproject'
src = f'{PROJECT}/generation.py'
assert os.path.exists(src), 'Upload generation.py into your MOSESproject folder on Drive first.'
shutil.copy(src, '/content/generation.py')
if '/content' not in sys.path:
    sys.path.insert(0, '/content')
print('generation.py staged at /content/generation.py')

## 6. Smoke test

One condition, one run, one seed, 20 draws. This validates adapter loading, the SELFIES tokenizer round-trip, property computation, and the output paths **before** committing hours to the full run. Watch for validity near 100%; a low value points at a tokenizer/decoding problem.

In [ ]:
import importlib
import generation as g
importlib.reload(g)  # reload to clear any stale module cache

# minimal overrides for the smoke test only
g.N_PER_CELL = 20
g.GENERATION_SEEDS = [201]
g.RUN_INDICES = [0]
g.CONDITIONS = {'mpt_standard': {'type': 'peft', 'prefix': 'mpt_standard_run'}}
g.MOSES_HELDOUT_CSV = None  # skip novelty for the quick test

g.main()

In [ ]:
# inspect the smoke-test output
import pandas as pd
p = '/content/drive/MyDrive/MOSESproject/results/generation_v2/mpt_standard_run0/gen201.csv'
df = pd.read_csv(p)
print('rows:', len(df), '  validity:', round(df['valid'].mean(), 3))
df[['canonical_smiles', 'QED', 'Fsp3', 'Solubility_logmolL', 'degenerate']].head(10)

## 7. Full run

Runs every condition as configured in `generation.py`: 4 conditions × 3 runs × 10 seeds × 1,000 draws = 120,000 molecules total (30,000 per condition).

This loops **per condition** so GPU memory is freed between conditions and a disconnect costs at most one cell. The run is **resumable**: completed `gen<seed>.csv` cells are skipped on rerun, so just re-run this cell after any Colab timeout.

In [ ]:
import importlib
import generation as g

FULL = {
    'finetuned_standard': {'type': 'full', 'prefix': 'standard_run'},
    'finetuned_weighted': {'type': 'full', 'prefix': 'weighted_run'},
    'mpt_standard':       {'type': 'peft', 'prefix': 'mpt_standard_run'},
    'mpt_weighted':       {'type': 'peft', 'prefix': 'mpt_weighted_run'},
}

for cond, spec in FULL.items():
    importlib.reload(g)                 # fresh module state per condition
    g.CONDITIONS = {cond: spec}         # run one condition at a time
    # g.MOSES_HELDOUT_CSV = '/content/drive/MyDrive/MOSESproject/moses_test.csv'
    print(f'\n########## {cond} ##########')
    g.main()

## 8. Check the summary

One row per (model version, generation seed) with N Valid, Validity %, N Unique, held-out uniqueness, and the mean/SD panel. The per-molecule CSVs under each `results/generation_v2/<model_version>/` folder feed the Tanimoto and FCD analysis later.

In [ ]:
import pandas as pd
s = pd.read_csv('/content/drive/MyDrive/MOSESproject/results/generation_v2/generation_summary.csv')
print(s.shape)
s.head(20)